In [ ]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv('D:\Powerpulse\cleaned_data.csv')

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Sivabarani M\AppData\Local\Temp\ipykernel_7676\216726128.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv('D:\Powerpulse\cleaned_data.csv')


In [5]:
df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Time_of_day,Day_type,Season,Day,Month,Year,Hour,Minute,Second,Rolling_avg_power,Daily_avg_power
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,Afternoon,Weekend,Winter,16,12,2006,17,24,0,4.216000,3.053475
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,Afternoon,Weekend,Winter,16,12,2006,17,25,0,4.788000,3.053475
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,Afternoon,Weekend,Winter,16,12,2006,17,26,0,4.983333,3.053475
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,Afternoon,Weekend,Winter,16,12,2006,17,27,0,5.084500,3.053475
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,Afternoon,Weekend,Winter,16,12,2006,17,28,0,4.800800,3.053475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2049275,0.946,0.000,240.43,4.0,0.0,0.0,0.0,Evening,Weekday,Fall,26,11,2010,20,58,0,1.173367,1.178230
2049276,0.944,0.000,240.00,4.0,0.0,0.0,0.0,Evening,Weekday,Fall,26,11,2010,20,59,0,1.163700,1.178230
2049277,0.938,0.000,239.82,3.8,0.0,0.0,0.0,Night,Weekday,Fall,26,11,2010,21,0,0,1.155067,1.178230
2049278,0.934,0.000,239.70,3.8,0.0,0.0,0.0,Night,Weekday,Fall,26,11,2010,21,1,0,1.144733,1.178230


# EDA

In [6]:
df.columns

Index(['Global_active_power', 'Global_reactive_power', 'Voltage',
       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3', 'Time_of_day', 'Day_type', 'Season', 'Day', 'Month',
       'Year', 'Hour', 'Minute', 'Second', 'Rolling_avg_power',
       'Daily_avg_power'],
      dtype='object')

In [7]:
# To find the average usage of energy as per daytype and timeofday
agg_df = df.groupby(['Day_type', 'Time_of_day'])['Global_active_power'].mean().reset_index()

fig = px.bar(agg_df, x='Day_type', y='Global_active_power', color='Time_of_day', barmode='group')
fig.show()

Insights

Weekdays
1. Energy consumption is highest in the Evening.
2. Morning and Night also show moderate usage, while Afternoon and Early Morning are relatively low.
3. This suggests people are most active at home in the Evening after work, leading to more appliance use.

Weekends
1. Again, Evenings record the highest consumption, even higher than weekday evenings.
2. Night usage is also higher on weekends compared to weekdays — possibly due to late-night activities.
3. Afternoons are moderate, while Early Mornings remain the lowest.

In [8]:
agg_df_2 = df.groupby(['Day_type', 'Time_of_day'], as_index=False)[
    ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
].mean()

sub_df = agg_df_2.melt(
    id_vars=['Day_type', 'Time_of_day'],
    value_vars=['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'],
    var_name='Sub_metering',
    value_name='Avg_energy'
)

fig = px.bar(
    sub_df,
    x='Day_type',
    y='Avg_energy',
    color='Sub_metering',
    facet_col='Time_of_day',
    barmode='stack',
    title='Average Energy Consumption by Day Type, Time of Day, and Sub-metering'
)
fig.show()

Insights from the chart:

1. Evening time dominates energy usage

Both Weekdays and Weekends show the highest consumption in the Evening, mainly driven by Sub_metering 3 (AC & water heater).

Suggestion: Customers should try to reduce evening peak loads (e.g., shift water heating or AC usage).

2. Weekends generally consume more than weekdays

Across Afternoon and Evening, Weekend bars are consistently taller than Weekday bars.

Likely because people stay at home more, using kitchen appliances (Sub_metering 1) and laundry devices (Sub_metering 2).

3. Sub_metering 3 is the largest contributor

Across all times, AC & water heater usage (Sub_metering 3) dominates total consumption.

This suggests heating/cooling is the primary driver of household electricity demand.

4. Kitchen & laundry appliances show smaller, but noticeable patterns

Sub_metering 1 (kitchen appliances) has spikes in Morning and Evening (possibly cooking times).

Sub_metering 2 (laundry, fridge, dryer) adds a steady contribution, slightly higher on Weekends.

In [9]:
agg_season = df.groupby(['Season', 'Day_type', 'Time_of_day'])[
    ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
].mean().reset_index()

sub_df = agg_season.melt(
    id_vars=['Season','Day_type','Time_of_day'],
    value_vars=['Sub_metering_1','Sub_metering_2','Sub_metering_3'],
    var_name='Sub_metering',
    value_name='Avg_energy'
)

fig = px.bar(
    sub_df,
    x='Season',
    y='Avg_energy',
    color='Sub_metering',
    facet_col='Time_of_day',
    facet_row='Day_type',   # 👈 Weekday/Weekend in separate rows
    barmode='stack',
    title="Energy Usage by Season, Day Type, and Time of Day"
)
fig.show()

Insights
1. Evenings are consistently the peak consumption period
	• Across all seasons and both Weekdays & Weekends, the Evening time-of-day shows the highest average energy usage.
	• This is mainly driven by Sub_metering_3 (AC & Water Heater), which dominates the stacked bar.
Suggestion: Customers should reduce heavy appliance use in the Evening, especially heating/cooling.

2. Winter has the highest energy demand
	• Winter Weekends (Afternoon & Evening) stand out with the tallest bars, showing higher heating demand (Sub_metering_3).
	• Winter Weekdays also see more Morning usage, likely from heating appliances.
Insight: Winter energy-saving measures (insulation, reduced heater use) would have the biggest impact.


3. Summer shows relatively lower average usage
	• Unlike Winter, Summer bars are shorter across all times, meaning less heating load.
	• Still, Evening usage remains the highest, likely due to AC usage (Sub_metering_3).
Suggestion: Encourage shifting AC use to non-peak hours or using fans in Summer evenings.


4. Sub-metering breakdown by appliance category
	• Sub_metering 3 (AC & Water Heater) is the largest contributor across all seasons and times.
	• Sub_metering 1 (Kitchen) is higher in Morning & Evening, matching cooking times.
	• Sub_metering 2 (Laundry/Fridge/Dryer) is moderate, slightly higher on Weekends, when people do household chores.
This shows that heating/cooling dominates, while kitchen & laundry add secondary peaks.


5. Weekends vs Weekdays
	• Weekends consistently consume more energy than Weekdays, especially Afternoons and Evenings.
	• Likely because people are at home more often, using kitchen appliances (Sub_metering_1) and laundry (Sub_metering_2) in addition to heating/cooling.
Customers should be encouraged to spread laundry & heavy appliance usage across weekdays where possible.
